In [1]:
%cd ../..

/Users/macos/Uni/1st_year/period_2/IntroML/homework


In [124]:
from pathlib import Path
from typing import Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
from sklearn import preprocessing
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from tqdm import trange

In [3]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

In [127]:
path_root = "E2/data_E2/"

def load_train(n: int = 8) -> pd.DataFrame:
    path_train = Path(path_root) / f"toy_train_{n}.csv"

    assert path_train.exists(), f"Path not existed: {path_train}"

    df = pd.read_csv(path_train)

    return df

In [5]:
df_test = pd.read_csv(Path(path_root) / "toy_test.csv")

In [56]:
df_train = load_train(4096)

In [57]:
df_train.head()

,x1,x2,y
0,1.278150,-1.735690,0
1,0.622329,1.131573,1
2,1.303741,0.035539,0
3,-0.143972,1.327961,1
4,0.323019,0.312350,1


In [16]:
def get_Xy_train_test(df_train: pd.DataFrame, df_test: pd.DataFrame) -> tuple:
    Xtrain, ytrain = df_train.iloc[:, :2], df_train.iloc[:, 2]
    Xtest, ytest = df_test.iloc[:, :2], df_test.iloc[:, 2]

    return Xtrain, Xtest, ytrain, ytest

## Task 2

In [9]:
feat_inter = preprocessing.PolynomialFeatures(degree=2, interaction_only=True)

In [23]:
EPS = 1e-6

def calc_accu(ytrue: np.ndarray, ypred: np.ndarray):
    assert ytrue.shape == ypred.shape

    out = 1/len(ytrue) * (np.abs(ytrue - ypred) <= EPS).sum()

    return out

def calc_perplex(ytrue: np.ndarray, ypred_prob: np.ndarray):
    return np.exp(
        -np.sum(
            np.log(ypred_prob[np.arange(len(ytrue)), ytrue])
        ) / len(ytrue)
    )

In [122]:
def dummy_predict_prob(X: pd.DataFrame, ytrain: pd.Series):
    frac = ytrain.value_counts()[1] / len(ytrain)
    prob =  np.array([[1 - frac, frac]]*len(X), dtype=np.float32)

    return prob

def dummy_predict(X: pd.DataFrame, ytrain: pd.Series):
    prob = dummy_predict_prob(X, ytrain)[:, 1]
    ypred = (prob > 0.5).astype(np.int32)

    return ypred


def optimal_bayes_predict_prob(X: pd.DataFrame):
    odds = -1/2 - X['x1'] + 3/2 * X['x2'] + X['x1'] * X['x2'] / 3
    p = (1 / (1 + np.exp(-odds.to_numpy())))[:, None]
    prob = np.concatenate([1 - p, p], axis=-1)

    return prob

def optimal_bayes_predict(X: pd.DataFrame):
    prob = optimal_bayes_predict_prob(X)[:, 1]
    ypred = (prob >= 0.5).astype(np.int32)

    return ypred

In [136]:
results_accu, results_perp = [], []


for i in trange(3, 13, 1):
    n = 2**i
    name = f"{n}"

    df_train = load_train(n)
    Xtrain, Xtest, ytrain, ytest = get_Xy_train_test(df_train, df_test)

    nb = GaussianNB()
    logistic_nointer = LogisticRegression(penalty=None)
    logistic_inter = LogisticRegression(penalty=None)

    nb.fit(Xtrain, ytrain)
    logistic_nointer.fit(Xtrain, ytrain)
    logistic_inter.fit(feat_inter.fit_transform(Xtrain), ytrain)

    results_accu.append({
        'n': n,
        'NB': calc_accu(ytest, nb.predict(Xtest)),
        'LR': calc_accu(ytest, logistic_nointer.predict(Xtest)),
        'LRi': calc_accu(ytest, logistic_inter.predict(feat_inter.fit_transform(Xtest))),
        'OptimalBayes': calc_accu(ytest, optimal_bayes_predict(Xtest)),
        'Dummy': calc_accu(ytest, dummy_predict(Xtest, ytrain))
    })
    results_perp.append({
        'n': n,
        'NB': calc_perplex(ytest, nb.predict_proba(Xtest)),
        'LR': calc_perplex(ytest, logistic_nointer.predict_proba(Xtest)),
        'LRi': calc_perplex(ytest, logistic_inter.predict_proba(feat_inter.fit_transform(Xtest))),
        'OptimalBayes': calc_perplex(ytest, optimal_bayes_predict_prob(Xtest)),
        'Dummy': calc_perplex(ytest, dummy_predict_prob(Xtest, ytrain))
    })

100%|██████████| 10/10 [00:00<00:00, 39.53it/s]


In [140]:
df_results_accu = pd.DataFrame.from_records(results_accu)

df_results_accu

,n,NB,LR,LRi,OptimalBayes,Dummy
0,8,0.5973,0.6837,0.5759,0.7572,0.5688
1,16,0.6537,0.6334,0.5681,0.7572,0.4312
2,32,0.7037,0.7337,0.7210,0.7572,0.5688
3,64,0.6843,0.7531,0.7529,0.7572,0.5688
4,128,0.7444,0.7533,0.7463,0.7572,0.5688
5,256,0.7507,0.7500,0.7572,0.7572,0.5688
6,512,0.7525,0.7528,0.7570,0.7572,0.5688
7,1024,0.7476,0.7505,0.7559,0.7572,0.5688
8,2048,0.7518,0.7524,0.7576,0.7572,0.5688
9,4096,0.7513,0.7510,0.7569,0.7572,0.5688


In [139]:
df_results_perp = pd.DataFrame.from_records(results_perp)

df_results_perp

,n,NB,LR,LRi,OptimalBayes,Dummy
0,8,81.029145,1.804528,2.128684,1.63518,2.000000
1,16,2.557424,6.102342,18.204835,1.63518,2.139477
2,32,1.954326,1.717563,1.816252,1.63518,2.000000
3,64,1.892499,1.653345,1.656399,1.63518,1.981256
4,128,1.665193,1.657166,1.694784,1.63518,1.983638
5,256,1.645215,1.645612,1.638630,1.63518,1.981105
6,512,1.656468,1.650604,1.641805,1.63518,1.981119
7,1024,1.653585,1.645227,1.635427,1.63518,1.984056
8,2048,1.654582,1.645270,1.635268,1.63518,1.981121
9,4096,1.648145,1.645596,1.635236,1.63518,1.981116


## Task c

In [141]:
logistic_inter.coef_

array([[-0.23971043, -1.029602  ,  1.44861832,  0.34548237]])